# Data preprocessing: fingerprint0.txt to autoencoder input

This notebook walks through the bootstrap file one representation at a time. Raw-file parsing uses GUFU's shared prepare_dataset_files API, so the notebook does not introduce a second parser with different BSSID, filtering, or missing-value rules. Later cells expose the prepared vocabulary, RSSI matrix, observation mask, named AE normalization, and final tensor.

Run from the repository root. This notebook prepares and inspects data only; it does not train a model or write artifacts.

## Step 1 — Select the source and preprocessing policy

The source uses the GUFU database text format: each line starts with x,y, followed by space-separated AP records. The bootstrap settings below match gufu.py: keep all finite readings, keep empty scans, drop malformed rows, and retain the strongest duplicate reading for an AP within one scan.

In [ ]:
from pathlib import Path
import torch

from gufu_data import PreprocessingConfig, prepare_dataset_files

SOURCE_PATH = Path("input/GF/fingerprint0.txt")
SOURCE_ROOT = Path("input")

if not SOURCE_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {SOURCE_PATH}; run this notebook from the GUFU repository root."
    )

PREPROCESSING_CONFIG = PreprocessingConfig(
    dataset_id="GF",
    batch_id="fingerprint0",
    batch_role="bootstrap_reference",
    input_format="gufu_db",
    rssi_threshold_dbm=None,
    prune_randomized_bssids=False,
    malformed_record_policy="drop",
    empty_scan_policy="keep",
    duplicate_bssid_policy="strongest",
    normalization_mode="paper_ae",
)

print(f"Source: {SOURCE_PATH}")
print(f"Settings: {PREPROCESSING_CONFIG.model_dump()}")

## Step 2 — Preview the raw text

This is only a preview of the input lines. Actual parsing and validation happen through the shared API in the next step. For each AP record, the parser uses the full BSSID and RSSI fields; trailing standard-deviation and count fields in this dataset are not model inputs.

In [ ]:
raw_lines = SOURCE_PATH.read_text(encoding="utf-8").splitlines()

print(f"Physical lines: {len(raw_lines)}")
for line_number, line in enumerate(raw_lines[:2], start=1):
    preview = line if len(line) <= 360 else line[:360] + " ..."
    print(f"line {line_number}: {preview}")

## Step 3 — Parse and validate into the prepared dataset

prepare_dataset_files performs row parsing, full-BSSID canonicalization, configured filtering and duplicate handling, stable scan-ID assignment, and vocabulary alignment. Keeping this operation in the shared module ensures the notebook uses the same rules as the training pipeline.

In [ ]:
prepared = prepare_dataset_files(
    [SOURCE_PATH],
    PREPROCESSING_CONFIG,
    source_root=SOURCE_ROOT,
)

print(prepared.summary())

## Step 4 — Inspect retained scans and dropped rows

Coordinates and provenance stay attached to scans, but the autoencoder uses only RSSI. source_row identifies the original one-based line; it is not inferred from coordinates, so repeated locations remain separate scans.

In [ ]:
print(f"Source files: {prepared.source_files}")
print(f"Preparation counts: {prepared.stats.model_dump()}")
print(f"Retained scans: {len(prepared.scans)}")
print(f"Dropped rows: {len(prepared.dropped_records)}")

for dropped in prepared.dropped_records[:5]:
    print(
        f"dropped {dropped.source_file}:{dropped.source_row}: "
        f"{dropped.reason}"
    )

first_scan = prepared.scans[0]
print(f"First scan ID: {first_scan.scan_id}")
print(f"Source row: {first_scan.source.source_file}:{first_scan.source.source_row}")
print(f"Coordinates available: {first_scan.has_coordinates}")
print(f"Coordinate role: {first_scan.coordinate_role}")

## Step 5 — Establish the ordered AP vocabulary

Every matrix column is identified by one full canonical BSSID. The vocabulary order is stable and shared across scans; ap_ids are the graph-facing IDs corresponding to the same columns.

In [ ]:
bssid_vocabulary = prepared.bssid_vocabulary
ap_ids = prepared.ap_ids
column_for_bssid = {
    bssid: column for column, bssid in enumerate(bssid_vocabulary)
}

print(f"Number of AP columns: {len(bssid_vocabulary)}")
print("First vocabulary entries:")
for column, (bssid, ap_id) in enumerate(zip(bssid_vocabulary[:12], ap_ids[:12])):
    print(f"column {column:3d}: {bssid} -> {ap_id}")

## Step 6 — Inspect one scan in the common AP space

A scan stores one raw dBm value per vocabulary entry. A False observation mask means the AP was not measured in that scan; its raw storage value is -120 dBm. The mask, rather than the numeric value, records whether a measurement was present.

In [ ]:
first_scan = prepared.scans[0]
first_scan_rows = list(
    zip(
        bssid_vocabulary,
        first_scan.rss_dbm,
        first_scan.observation_mask,
        strict=True,
    )
)
observed_rows = [row for row in first_scan_rows if row[2]]
missing_rows = [row for row in first_scan_rows if not row[2]]

print(f"Observed APs in first scan: {len(observed_rows)}")
print(f"Missing AP slots in first scan: {len(missing_rows)}")
print("First observed entries (BSSID, RSSI dBm, observed):")
for row in observed_rows[:10]:
    print(row)
print("First missing entries:")
for row in missing_rows[:5]:
    print(row)

## Step 7 — Align bootstrap coordinates with scan rows

The AE uses RSSI vectors, while localization training also needs coordinates. Bootstrap training positions are stored separately from scans and are joined here by stable scan ID, never by coordinate value or assumed list order. Only reference_positions are used; evaluation_positions are not training labels. The explicit mask keeps an unlabeled row distinct from a valid location such as (0, 0).

In [ ]:
import pandas as pd
from IPython.display import display

scan_ids = tuple(scan.scan_id for scan in prepared.scans)
reference_xy_by_scan_id = {
    position.scan_id: (position.x, position.y)
    for position in prepared.reference_positions
}

position_xy = torch.zeros((len(prepared.scans), 2), dtype=torch.float32)
position_label_mask = torch.zeros(len(prepared.scans), dtype=torch.bool)

for row_index, scan in enumerate(prepared.scans):
    xy = reference_xy_by_scan_id.get(scan.scan_id)
    if xy is not None:
        position_xy[row_index] = torch.tensor(xy, dtype=torch.float32)
        position_label_mask[row_index] = True

expected_reference_ids = {
    scan.scan_id
    for scan in prepared.scans
    if scan.coordinate_role == "bootstrap_train"
}
assert set(reference_xy_by_scan_id) == expected_reference_ids
assert len(scan_ids) == len(set(scan_ids))
assert position_xy.shape == (len(scan_ids), 2)
assert position_label_mask.shape == (len(scan_ids),)

coordinate_table = []
for row_index, scan in enumerate(prepared.scans):
    coordinate_table.append(
        {
            "row": row_index,
            "scan_id": scan.scan_id,
            "source_row": scan.source.source_row,
            "x": (
                float(position_xy[row_index, 0])
                if position_label_mask[row_index]
                else None
            ),
            "y": (
                float(position_xy[row_index, 1])
                if position_label_mask[row_index]
                else None
            ),
            "label_available": bool(position_label_mask[row_index]),
            "role": scan.coordinate_role,
        }
    )

print(
    f"Position tensor shape: {tuple(position_xy.shape)}; "
    f"labeled rows: {int(position_label_mask.sum())}/{len(scan_ids)}"
)
coordinate_frame = pd.DataFrame(coordinate_table).set_index("row")
display(coordinate_frame)

## Step 8 — Assemble the raw RSSI matrix and mask

Rows follow prepared.scans; columns follow the ordered vocabulary. This is still raw dBm, before AE normalization.

In [ ]:
raw_rssi_dbm = torch.tensor(
    [scan.rss_dbm for scan in prepared.scans],
    dtype=torch.float32,
)
observation_mask = torch.tensor(
    [scan.observation_mask for scan in prepared.scans],
    dtype=torch.bool,
)

assert raw_rssi_dbm.shape == observation_mask.shape
assert raw_rssi_dbm.shape == (len(prepared.scans), len(bssid_vocabulary))

print(f"Raw RSSI shape: {tuple(raw_rssi_dbm.shape)}")
print(f"Observation mask shape: {tuple(observation_mask.shape)}")
print(f"First raw row, first 12 columns: {raw_rssi_dbm[0, :12].tolist()}")
print(f"First mask row, first 12 columns: {observation_mask[0, :12].tolist()}")

## Step 9 — Normalize RSSI for the autoencoder

The selected paper_ae conversion maps observed values with (r + 120) / 120. An unobserved slot maps to 0.0, using the observation mask explicitly. This is the shared named conversion, not a value-based missingness guess.

In [ ]:
ae_rows = prepared.ae_matrix()
ae_input = torch.tensor(ae_rows, dtype=torch.float32)

assert ae_input.shape == raw_rssi_dbm.shape
assert torch.isfinite(ae_input).all()

observed_indices = torch.nonzero(observation_mask[0], as_tuple=False).flatten()
if len(observed_indices):
    column = int(observed_indices[0])
    raw_value = raw_rssi_dbm[0, column].item()
    ae_value = ae_input[0, column].item()
    print(
        f"Example measured value: {raw_value:.2f} dBm -> {ae_value:.6f} "
        f"= ({raw_value:.2f} + 120) / 120"
    )

print(f"Unobserved AE values: {torch.unique(ae_input[~observation_mask]).tolist()}")
print(
    f"AE input range: {ae_input.min().item():.6f} to "
    f"{ae_input.max().item():.6f}"
)

## Step 10 — Final autoencoder input

ae_input is the two-dimensional tensor the AE receives: [number of scans, number of APs]. Its columns must stay paired with bssid_vocabulary whenever the tensor is saved or passed to another stage.

In [ ]:
assert ae_input.ndim == 2
assert ae_input.shape[0] == len(prepared.scans)
assert ae_input.shape[1] == len(prepared.bssid_vocabulary)
assert len(set(scan.scan_id for scan in prepared.scans)) == len(prepared.scans)
assert torch.all(ae_input[~observation_mask] == 0.0)
assert torch.allclose(
    ae_input[observation_mask],
    (raw_rssi_dbm[observation_mask] + 120.0) / 120.0,
)

print(f"Ready for AE: {tuple(ae_input.shape)}")
print(f"Example scan ID: {prepared.scans[0].scan_id}")
print(f"Vocabulary width: {len(prepared.bssid_vocabulary)}")
print(f"Tensor dtype: {ae_input.dtype}")
assert len(scan_ids) == len(ae_input)
assert position_xy.shape[0] == ae_input.shape[0]
print(f"Coordinates aligned to RSS row 0: {coordinate_table[0]}")